# Image Tagging Agent -- Test Notebook

In [ ]:
# Add project root to path so "import src..." works (run this cell first)
import sys
from pathlib import Path

cwd = Path.cwd()
if (cwd / "src").is_dir():
    repo_root = cwd
else:
    # Kernel cwd is e.g. tests/notebooks — go up to project root
    repo_root = cwd.parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print("Project root:", repo_root)

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
print("OPENAI_MODEL:", os.getenv("OPENAI_MODEL"))

## Phase 0: Import Verification

In [ ]:
import src.image_tagging.image_tagging
print("All imports OK")

## Phase 1: Vision Analysis

Run the graph with an image (file path or base64). Requires `OPENAI_API_KEY` in `.env`.

In [ ]:
# Optional: set path to a local image (JPG/PNG/WEBP). If unset, we use a tiny 1x1 PNG base64.
# In Jupyter we use top-level await (no asyncio.run) because the kernel already has a running event loop.
import base64
from pathlib import Path
from src.image_tagging.image_tagging import graph

IMAGE_PATH = None  # e.g. Path("path/to/your/image.jpg")

if IMAGE_PATH and Path(IMAGE_PATH).exists():
    raw = Path(IMAGE_PATH).read_bytes()
    image_base64 = base64.b64encode(raw).decode("utf-8")
else:
    # Minimal 1x1 PNG for import/graph test (replace with real image for full vision output)
    image_base64 = "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8z8BQDwAEhQGAhKmMIQAAAABJRU5ErkJggg=="

state = {
    "image_id": "notebook-test",
    "image_url": "",
    "image_base64": image_base64,
    "metadata": {},
    "vision_description": "",
    "vision_raw_tags": {},
    "partial_tags": [],
    "processing_status": "pending",
}

result = await graph.ainvoke(state)
print("Status:", result.get("processing_status"))
print("Error:", result.get("error"))
if result.get("vision_raw_tags"):
    print("Vision raw_tags keys:", list(result["vision_raw_tags"].keys()))
if result.get("vision_description"):
    print("Vision description (preview):", (result["vision_description"] or "")[:200])

## Phase 2: Season Tagger

The graph now runs preprocess → vision → season_tagger. Check `partial_tags` and season tags.

In [ ]:
# After running the Phase 1 cell above, inspect tagger output:
partial = result.get("partial_tags") or []
print("partial_tags:", partial)
season_result = next((p for p in partial if p.get("category") == "season"), None)
if season_result:
    print("Season tags:", season_result.get("tags"))
    print("Season confidence_scores:", season_result.get("confidence_scores"))
else:
    print("No season result (vision may have failed or graph not run yet)")

## Phase 3: All 8 Parallel Taggers

The graph runs all 8 taggers in parallel (Send API). Inspect `partial_tags` for every category: season, theme, objects, dominant_colors, design_elements, occasion, mood, product_type.

In [ ]:
# Build tags_by_category from result (same shape the API returns)
partial = result.get("partial_tags") or []
tags_by_category = {}
for p in partial:
    cat = p.get("category")
    if cat:
        tags_by_category[cat] = {"tags": p.get("tags") or [], "confidence_scores": p.get("confidence_scores") or {}}
for cat in ["season", "theme", "objects", "dominant_colors", "design_elements", "occasion", "mood", "product_type"]:
    data = tags_by_category.get(cat, {})
    print(f"{cat}: tags={data.get('tags', [])} confidence={data.get('confidence_scores', {})}")

## Phase 4: Validation and Confidence

After the 8 taggers, the graph runs tag_validator → confidence_filter → tag_aggregator. Inspect `validated_tags`, `flagged_tags`, `tag_record`, and `needs_review`.

In [ ]:
# Phase 4 outputs (run Phase 1 cell first to have result)
print("needs_review:", result.get("needs_review"))
print("processing_status:", result.get("processing_status"))
validated = result.get("validated_tags") or {}
print("validated_tags categories:", list(validated.keys()))
flagged = result.get("flagged_tags") or []
print("flagged_tags count:", len(flagged))
for f in flagged[:5]:
    print("  ", f)
tr = result.get("tag_record")
if tr:
    print("tag_record keys:", list(tr.keys()))
    print("tag_record (sample):", {k: tr.get(k) for k in ["season", "theme", "mood"] if tr.get(k)})

## Phase 5: Supabase Integration

When `DATABASE_URL` is set, the server upserts the tag record after analyze. Here we call the Supabase client directly to list recent images and get one record by `image_id`. (The API also exposes `GET /api/tag-images` and `GET /api/tag-image/{id}`.)

In [ ]:
# Optional: requires DATABASE_URL in .env and image_tags table created (see sql/image_tags.sql)
from src.services.supabase.settings import SUPABASE_ENABLED

if SUPABASE_ENABLED:
    from src.services.supabase.client import list_tag_images, get_tag_record
    items = list_tag_images(limit=5, offset=0)
    print("Recent tag_images (limit 5):", len(items))
    for row in items[:3]:
        print(" ", row.get("image_id"), row.get("processing_status"), (row.get("tag_record") or {}).get("season"))
    # Get the record we just created (if Phase 1 was run with result["image_id"])
    image_id = result.get("image_id")
    if image_id:
        rec = get_tag_record(image_id)
        print("get_tag_record for notebook-test:", "found" if rec else "not found (run server + analyze to persist)")
else:
    print("Supabase not enabled (DATABASE_URL not set). Set it and run the server to persist; then GET /api/tag-images.")

## Phase 6: Search Page and Taxonomy

Taxonomy is in `src/image_tagging/taxonomy.py`. The API exposes `GET /api/taxonomy`, `GET /api/search-images`, `GET /api/available-filters`. Here we load taxonomy from the module and, if Supabase is enabled, run filtered search and available-filters.

In [ ]:
# Taxonomy from module (same structure as GET /api/taxonomy)
from src.image_tagging import taxonomy as tax
print("Flat categories: season (%d), theme (%d), design_elements (%d), occasion (%d), mood (%d)" % (
    len(tax.SEASON), len(tax.THEME), len(tax.DESIGN), len(tax.OCCASION), len(tax.MOOD)))
print("Hierarchical: objects groups:", list(tax.OBJECTS_CATEGORIES.keys()))
print("Hierarchical: dominant_colors groups:", list(tax.COLORS_CATEGORIES.keys()))

# Search and available filters (requires DATABASE_URL and image_tags table)
from src.services.supabase.settings import SUPABASE_ENABLED
if SUPABASE_ENABLED:
    from src.services.supabase.client import search_images_filtered, get_available_filter_values
    items = search_images_filtered({}, limit=5, offset=0)
    print("search_images_filtered (no filters) count:", len(items))
    avail = get_available_filter_values(None)
    print("available_filters categories:", list(avail.keys()) if isinstance(avail, dict) else "N/A")
else:
    print("Supabase not enabled. Start server and use GET /api/taxonomy, GET /api/search-images, GET /api/available-filters.")

## Phase 7: Bulk Upload

The server exposes `POST /api/bulk-upload` (multiple files) and `GET /api/bulk-status/{batch_id}`. Start the server (`python -m src.server`) and run the cell below to upload 2 small images and poll until complete.

In [ ]:
# Bulk upload via API (requires server running: python -m src.server)
import time
import httpx

API_BASE = "http://localhost:8000"  # change if server runs elsewhere
# Minimal 1x1 PNG bytes
PNG_BYTES = b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\x01\x00\x00\x00\x01\x08\x02\x00\x00\x00\x90wS\xde\x00\x00\x00\x0cIDATx\x9cc\xf8\x0f\x00\x00\x01\x01\x00\x05\x18\xd8N\x00\x00\x00\x00IEND\xaeB`\x82'

try:
    with httpx.Client(timeout=60.0) as client:
        files = [("files", ("a.png", PNG_BYTES, "image/png")), ("files", ("b.png", PNG_BYTES, "image/png"))]
        r = client.post(f"{API_BASE}/api/bulk-upload", files=files)
        r.raise_for_status()
        data = r.json()
        batch_id = data["batch_id"]
        total = data["total"]
        print("Bulk upload started:", batch_id, "total:", total)
        while True:
            r2 = client.get(f"{API_BASE}/api/bulk-status/{batch_id}")
            r2.raise_for_status()
            st = r2.json()
            n_done = len(st.get("results", []))
            print("  status:", st["status"], "processed:", n_done, "of", st["total"])
            if st["status"] == "complete":
                break
            time.sleep(1)
        print("Bulk batch complete. Results:", st.get("results", []))
except httpx.ConnectError as e:
    print("Server not reachable. Start with: python -m src.server")
except Exception as e:
    print("Error:", e)